<a href="https://colab.research.google.com/github/jsemprini/Iowa-Water-Nitrate-Births-8488update/blob/main/Copy_of_3_link_water_birth_final_T1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# File 3 (Final V3) — Link births to completed and observed T1 nitrate

**Purpose.** Link each pregnancy's fixed 98-day T1 window (gestational days 0–97) to the V3 county-quarter water panel using exact overlap-day weighting.

Only two exposure variables are carried into the analysis dataset:

- `t1_mean_complete`: day-weighted T1 mean using the completed county-quarter panel (observed values where available; V3 imputed values otherwise).
- `t1_mean_observed`: day-weighted T1 mean using observed county-quarter nitrate only; missing if no T1 day falls in an observed county-quarter.

T1 exposure-quality statistics are calculated and saved as separate QA tables but are not retained as analysis variables.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


Mounted at /content/drive


In [ ]:
# ============================================================
# 0. PATHS AND SETTINGS
# ============================================================
import os
import gc
import numpy as np
import pandas as pd

BIRTH_PATH_CSV = (
    '/content/drive/MyDrive/plos-update-v3/2-birth/'
    'births_linkage_T1_v3.csv'
)
WATER_PATH_CSV = (
    '/content/drive/MyDrive/plos-update-v3/1-water/'
    'final_county_quarter_complete_1982_1988_v3.csv'
)
OUTPUT_DIR = '/content/drive/MyDrive/plos-update-v3/3-linked'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Birth input:', BIRTH_PATH_CSV)
print('Water input:', WATER_PATH_CSV)


Birth input: /content/drive/MyDrive/plos-update-v3/2-birth/births_linkage_T1_v3.csv
Water input: /content/drive/MyDrive/plos-update-v3/1-water/final_county_quarter_complete_1982_1988_v3.csv


In [ ]:
# ============================================================
# 1. LOAD AND VALIDATE
# ============================================================
births = pd.read_csv(BIRTH_PATH_CSV, low_memory=False)
water = pd.read_csv(WATER_PATH_CSV, low_memory=False)

births['county_fips'] = pd.to_numeric(births['county_fips'], errors='coerce').astype(np.int64)
for c in ['t1_start', 't1_end']:
    births[c] = pd.to_datetime(births[c], errors='coerce')

required_birth_cols = [
    'birth_id', 'county_fips', 'birth_year', 'gest_age_weeks',
    'conception_quarter', 'birthweight_g', 'infant_male', 'maternal_age',
    'maternal_race_broad', 'married', 'prenatal_by5', 'live_birth_order',
    'maternal_education_years', 't1_start', 't1_end'
]
missing_birth = [c for c in required_birth_cols if c not in births.columns]
if missing_birth:
    raise KeyError('Required birth columns missing: ' + ', '.join(missing_birth))

required_water_cols = [
    'county_fips', 'quarter_start', 'quarter_end',
    'nitrate_complete', 'nitrate_observed', 'exposure_source'
]
missing_water = [c for c in required_water_cols if c not in water.columns]
if missing_water:
    raise KeyError('Required water columns missing: ' + ', '.join(missing_water))

water['county_fips'] = pd.to_numeric(water['county_fips'], errors='coerce').astype(np.int64)
water['nitrate_complete'] = pd.to_numeric(water['nitrate_complete'], errors='coerce')
water['nitrate_observed'] = pd.to_numeric(water['nitrate_observed'], errors='coerce')
water['exposure_source'] = water['exposure_source'].astype('string').str.strip().str.lower()
water['quarter_start'] = pd.to_datetime(water['quarter_start'], errors='coerce')
water['quarter_end'] = pd.to_datetime(water['quarter_end'], errors='coerce')
water['year'] = water['quarter_start'].dt.year.astype(np.int16)
water['quarter'] = water['quarter_start'].dt.quarter.astype(np.int8)
water['_qindex'] = water['year'].astype(np.int64) * 4 + water['quarter'].astype(np.int64)

assert births['birth_id'].is_unique
assert births['gest_age_weeks'].between(20, 41).all()
assert ((births['t1_end'] - births['t1_start']).dt.days == 97).all()
assert water['nitrate_complete'].notna().all()
assert (water['nitrate_complete'] >= 0).all()
assert water[['county_fips', '_qindex']].duplicated().sum() == 0

if set(births['county_fips']) - set(water['county_fips']):
    raise ValueError('At least one birth county is absent from the water panel.')

print('Birth rows:', f'{len(births):,}')
print('Water rows:', f'{len(water):,}')


Birth rows: 164,977
Water rows: 2,772


In [ ]:
# ============================================================
# 2. EXACT-OVERLAP T1 LINKAGE
# ============================================================
def quarter_index_from_date(date_series):
    years = date_series.dt.year.to_numpy(dtype=np.int64)
    months = date_series.dt.month.to_numpy(dtype=np.int64)
    quarters = (months - 1) // 3 + 1
    return years * 4 + quarters

base = births[['birth_id', 'county_fips', 't1_start', 't1_end']].copy()
base = base.rename(columns={'t1_start': 'window_start', 't1_end': 'window_end'})
valid = (
    base['window_start'].notna()
    & base['window_end'].notna()
    & base['window_start'].le(base['window_end'])
)
if not valid.all():
    raise ValueError('Invalid T1 start/end detected in linkage input.')

start_qi = quarter_index_from_date(base['window_start'])
end_qi = quarter_index_from_date(base['window_end'])
n_quarters = (end_qi - start_qi + 1).astype(np.int64)
total_rows = int(n_quarters.sum())
pos = np.repeat(np.arange(len(base), dtype=np.int64), n_quarters)
group_starts = np.repeat(np.cumsum(n_quarters) - n_quarters, n_quarters)
offsets = np.arange(total_rows, dtype=np.int64) - group_starts

long = pd.DataFrame({
    'birth_id': base['birth_id'].to_numpy()[pos],
    'county_fips': base['county_fips'].to_numpy(dtype=np.int64)[pos],
    'window_start': base['window_start'].to_numpy()[pos],
    'window_end': base['window_end'].to_numpy()[pos],
    '_qindex': start_qi[pos] + offsets,
})

water_cols = [
    'county_fips', '_qindex', 'quarter_start', 'quarter_end',
    'nitrate_complete', 'nitrate_observed', 'exposure_source'
]
long = long.merge(
    water[water_cols],
    on=['county_fips', '_qindex'],
    how='left',
    validate='m:1'
)

if long['nitrate_complete'].isna().any():
    bad = long.loc[long['nitrate_complete'].isna()].head()
    raise ValueError(f'Missing completed nitrate during T1. Example:\n{bad}')

long['overlap_start'] = long[['window_start', 'quarter_start']].max(axis=1)
long['overlap_end'] = long[['window_end', 'quarter_end']].min(axis=1)
long['overlap_days'] = (long['overlap_end'] - long['overlap_start']).dt.days + 1
if (long['overlap_days'] <= 0).any():
    raise ValueError('Nonpositive T1 overlap days detected.')

complete = long['nitrate_complete'].to_numpy(dtype=float)
observed_value = long['nitrate_observed'].to_numpy(dtype=float)
days = long['overlap_days'].to_numpy(dtype=float)
observed = np.isfinite(observed_value)

long['_complete_weighted'] = complete * days
long['_observed_weighted'] = np.where(observed, observed_value * days, 0.0)
long['_obsdays'] = np.where(observed, days, 0.0)
long['_impdays'] = np.where(observed, 0.0, days)

agg = (
    long.groupby('birth_id', sort=False)
    .agg(
        t1_days=('overlap_days', 'sum'),
        t1_nquarters=('_qindex', 'size'),
        _complete_num=('_complete_weighted', 'sum'),
        _observed_num=('_observed_weighted', 'sum'),
        t1_obsdays=('_obsdays', 'sum'),
        t1_impdays=('_impdays', 'sum'),
    )
    .reset_index()
)

agg['t1_mean_complete'] = agg['_complete_num'] / agg['t1_days']
agg['t1_mean_observed'] = np.where(
    agg['t1_obsdays'] > 0,
    agg['_observed_num'] / agg['t1_obsdays'],
    np.nan
)
agg['t1_obsfrac'] = agg['t1_obsdays'] / agg['t1_days']
agg['t1_anyimp'] = (agg['t1_impdays'] > 0).astype(np.int8)
agg['t1_allobs'] = (agg['t1_impdays'] == 0).astype(np.int8)

# Every final pregnancy should contribute exactly 98 T1 days.
if not agg['t1_days'].eq(98).all():
    bad = agg.loc[~agg['t1_days'].eq(98)].head()
    raise ValueError(f'T1 overlap did not sum to 98 days for all births. Example:\n{bad}')

print('T1 linkage complete for births:', f'{len(agg):,}')
print('Mean completed T1 nitrate:', agg['t1_mean_complete'].mean())
print('Mean observed-only T1 nitrate:', agg['t1_mean_observed'].mean())


T1 linkage complete for births: 164,977
Mean completed T1 nitrate: 3.46736869646248
Mean observed-only T1 nitrate: 3.5643058642856507


In [ ]:
# ============================================================
# 3. T1 EXPOSURE-QUALITY QA
# ============================================================
t1_quality_qa = pd.DataFrame({
    'metric': [
        'n_births',
        'mean_t1_days',
        'mean_t1_mean_complete',
        'mean_t1_mean_observed_among_available',
        'pct_births_with_any_imputed_t1_days',
        'pct_births_with_all_t1_days_observed',
        'mean_fraction_t1_days_observed',
        'pct_births_with_no_observed_t1_days',
        'correlation_complete_vs_observed_among_available',
    ],
    'value': [
        len(agg),
        agg['t1_days'].mean(),
        agg['t1_mean_complete'].mean(),
        agg['t1_mean_observed'].mean(),
        100 * agg['t1_anyimp'].mean(),
        100 * agg['t1_allobs'].mean(),
        agg['t1_obsfrac'].mean(),
        100 * agg['t1_mean_observed'].isna().mean(),
        agg[['t1_mean_complete', 't1_mean_observed']].corr().iloc[0, 1],
    ]
})
display(t1_quality_qa)

# Distributional comparison of complete versus observed-only exposure.
t1_distribution_qa = pd.DataFrame({
    'measure': ['t1_mean_complete', 't1_mean_observed'],
    'n_nonmissing': [agg['t1_mean_complete'].notna().sum(), agg['t1_mean_observed'].notna().sum()],
    'mean': [agg['t1_mean_complete'].mean(), agg['t1_mean_observed'].mean()],
    'sd': [agg['t1_mean_complete'].std(), agg['t1_mean_observed'].std()],
    'p25': [agg['t1_mean_complete'].quantile(.25), agg['t1_mean_observed'].quantile(.25)],
    'median': [agg['t1_mean_complete'].median(), agg['t1_mean_observed'].median()],
    'p75': [agg['t1_mean_complete'].quantile(.75), agg['t1_mean_observed'].quantile(.75)],
    'min': [agg['t1_mean_complete'].min(), agg['t1_mean_observed'].min()],
    'max': [agg['t1_mean_complete'].max(), agg['t1_mean_observed'].max()],
})
display(t1_distribution_qa)

# Exposure quality by birth year; diagnostic only.
qa_merge = births[['birth_id', 'birth_year']].merge(
    agg[['birth_id', 't1_mean_complete', 't1_mean_observed', 't1_obsfrac', 't1_anyimp', 't1_allobs']],
    on='birth_id', how='left', validate='1:1'
)
t1_quality_by_birth_year = (
    qa_merge.groupby('birth_year', as_index=False)
    .agg(
        n=('birth_id', 'size'),
        mean_t1_complete=('t1_mean_complete', 'mean'),
        mean_t1_observed=('t1_mean_observed', 'mean'),
        mean_obsfrac=('t1_obsfrac', 'mean'),
        pct_anyimp=('t1_anyimp', 'mean'),
        pct_allobs=('t1_allobs', 'mean'),
    )
)
t1_quality_by_birth_year['pct_anyimp'] *= 100
t1_quality_by_birth_year['pct_allobs'] *= 100
display(t1_quality_by_birth_year)


,metric,value
0,n_births,164977.000000
1,mean_t1_days,98.000000
2,mean_t1_mean_complete,3.467369
3,mean_t1_mean_observed_among_available,3.564306
4,pct_births_with_any_imputed_t1_days,32.475436
5,pct_births_with_all_t1_days_observed,67.524564
6,mean_fraction_t1_days_observed,0.779176
7,pct_births_with_no_observed_t1_days,11.550095
8,correlation_complete_vs_observed_among_available,0.982730


,measure,n_nonmissing,mean,sd,p25,median,p75,min,max
0,t1_mean_complete,164977,3.467369,3.142897,0.911594,2.669388,5.317551,0.0,39.136735
1,t1_mean_observed,145922,3.564306,3.372649,0.670000,2.844439,5.760612,0.0,43.000000


,birth_year,n,mean_t1_complete,mean_t1_observed,mean_obsfrac,pct_anyimp,pct_allobs
0,1983,7951,2.897623,2.975913,0.869163,24.361716,75.638284
1,1984,33576,3.626771,3.793751,0.719748,38.884918,61.115082
2,1985,32799,3.139667,3.196492,0.805426,30.635080,69.364920
3,1986,31325,3.283039,3.412555,0.804961,28.897047,71.102953
4,1987,30387,3.722609,3.872641,0.757310,33.603186,66.396814
5,1988,28939,3.741890,3.775483,0.788702,32.043263,67.956737


In [ ]:
# ============================================================
# 4. CREATE COMPACT LINKED DATA + SAVE
# ============================================================
# Merge only the two analysis exposures. QA metrics remain in separate tables.
linked = births.merge(
    agg[['birth_id', 't1_mean_complete', 't1_mean_observed']],
    on='birth_id',
    how='left',
    validate='1:1'
)

if linked['t1_mean_complete'].isna().any():
    raise ValueError('t1_mean_complete is missing for at least one linked birth.')
if (linked['t1_mean_complete'] < 0).any():
    raise ValueError('Negative t1_mean_complete detected.')

# File 4A needs the two source variables below to create the final parity and
# maternal-education indicators; they are dropped in File 4A.
linked_keep = [
    'county_fips',
    'birth_year',
    'gest_age_weeks',
    'conception_quarter',
    'birthweight_g',
    'infant_male',
    'maternal_age',
    'maternal_race_broad',
    'married',
    'prenatal_by5',
    't1_mean_complete',
    't1_mean_observed',
    'live_birth_order',
    'maternal_education_years',
]
linked_compact = linked[linked_keep].copy()

CSV_PATH = os.path.join(OUTPUT_DIR, 'birth_water_linked_T1_v3.csv')
PARQUET_PATH = os.path.join(OUTPUT_DIR, 'birth_water_linked_T1_v3.parquet')
linked_compact.to_csv(CSV_PATH, index=False)
try:
    linked_compact.to_parquet(PARQUET_PATH, index=False)
except Exception as exc:
    print('Parquet save skipped:', exc)

t1_quality_qa.to_csv(os.path.join(OUTPUT_DIR, 't1_exposure_quality_qa_v3.csv'), index=False)
t1_distribution_qa.to_csv(os.path.join(OUTPUT_DIR, 't1_exposure_distribution_qa_v3.csv'), index=False)
t1_quality_by_birth_year.to_csv(os.path.join(OUTPUT_DIR, 't1_exposure_quality_by_birth_year_v3.csv'), index=False)

metadata = pd.DataFrame({
    'item': [
        'final_exposure_window',
        't1_mean_complete',
        't1_mean_observed',
        'linkage_weighting',
        'qa_retention',
    ],
    'value': [
        'gestational days 0-97 inclusive (98 days)',
        'overlap-day-weighted mean of nitrate_complete across T1',
        'overlap-day-weighted mean of nitrate_observed across observed T1 days only; missing if zero observed T1 days',
        'exact calendar-day overlap with county-quarter water values',
        'T1 imputation/observation quality metrics saved separately and not carried as analysis variables',
    ]
})
metadata.to_csv(os.path.join(OUTPUT_DIR, 'linkage_metadata_T1_v3.csv'), index=False)

del long, qa_merge
_gc = gc.collect()
print('File 3 Final V3 complete.')
print('Primary File 4A input:', PARQUET_PATH)


File 3 Final V3 complete.
Primary File 4A input: /content/drive/MyDrive/plos-update-v3/3-linked/birth_water_linked_T1_v3.parquet
